In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Macrophage reclustering + R/NR composition + DE (1 vs rest) using Scanpy.

Input:
  /ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/01_mapping_raw_scRNA_seq_to_reference/adata_scvi_integrated_all_cells.h5ad

Output dir:
  /ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/08_other_cell_types/myeloid/macrophage

What it does:
  1) Subset predicted_labels == "Macrophages"
  2) Neighbors/UMAP/Leiden (res=0.6) using X_scvi latent
  3) Count NR vs R per cluster (absolute cell counts) + barplots
  4) DE: leiden cluster vs rest using layer norm_expr_log1p, save per-cluster signature genes
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings("ignore")

# ----------------------------
# Matplotlib config (PDF friendly)
# ----------------------------
mpl.rcParams["font.family"] = "Liberation Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# ----------------------------
# Paths
# ----------------------------
H5AD_PATH = (
    "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/"
    "01_mapping_raw_scRNA_seq_to_reference/adata_scvi_integrated_all_cells.h5ad"
)

OUTDIR = (
    "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/"
    "08_other_cell_types/myeloid/macrophage"
)

FIGDIR = os.path.join(OUTDIR, "figures")
TABDIR = os.path.join(OUTDIR, "tables")
DEDIR  = os.path.join(OUTDIR, "de")
SIGDIR = os.path.join(OUTDIR, "signatures")
for d in [OUTDIR, FIGDIR, TABDIR, DEDIR, SIGDIR]:
    os.makedirs(d, exist_ok=True)

sc.settings.verbosity = 2

# ----------------------------
# Helper: robust column detection
# ----------------------------
def pick_obs_col(adata, candidates):
    lower_map = {c.lower(): c for c in adata.obs.columns}
    for cand in candidates:
        if cand in adata.obs.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def pick_scvi_rep(adata):
    # common keys: X_scVI, X_scvi, X_scANVI, X_scanvi, etc.
    keys = list(adata.obsm.keys())
    # prioritize exact matches
    for k in ["X_scVI", "X_scvi", "X_scanvi", "X_scANVI"]:
        if k in adata.obsm:
            return k
    # fallback: any key containing "scvi" or "scanvi"
    for k in keys:
        lk = k.lower()
        if "scvi" in lk or "scanvi" in lk:
            return k
    return None

def save_current_fig(path_png, path_pdf=None, dpi=220):
    plt.tight_layout()
    plt.savefig(path_png, dpi=dpi)
    if path_pdf is not None:
        plt.savefig(path_pdf)
    plt.close()

# ----------------------------
# Load
# ----------------------------
print(f"[1/6] Loading: {H5AD_PATH}")
ad = sc.read_h5ad(H5AD_PATH)

# ----------------------------
# Subset Macrophages
# ----------------------------
print("[2/6] Subsetting predicted_labels == 'Macrophages'")
label_col = pick_obs_col(ad, ["predicted_labels", "predicted_label", "celltype", "cell_type"])
if label_col is None:
    raise ValueError(
        "Cannot find predicted_labels column in adata.obs. "
        f"Available columns: {list(ad.obs.columns)[:50]} ... (total {ad.obs.shape[1]})"
    )

mac_label = "Macrophages"
if mac_label not in ad.obs[label_col].astype(str).unique():
    # show top labels for debugging
    top = ad.obs[label_col].astype(str).value_counts().head(30)
    raise ValueError(
        f"'{mac_label}' not found in obs['{label_col}'].\n"
        f"Top labels:\n{top}"
    )

ad_mac = ad[ad.obs[label_col].astype(str) == mac_label].copy()
print(f"Macrophages cells: {ad_mac.n_obs:,} | genes: {ad_mac.n_vars:,}")

# Save subset early
#subset_path = os.path.join(OUTDIR, "adata_macrophage_subset.h5ad")
#ad_mac.write(subset_path)
#print(f"Saved subset: {subset_path}")



In [ ]:
# ----------------------------
# Choose R/NR column
# ----------------------------
non_resp = {"PHD001", "PHD002", "PHD008"}
resp = {"PHD003", "PHD004", "PHD009"}
mapping = {p: "Non-Responder" for p in non_resp}
mapping.update({p: "Responder" for p in resp})

ad_mac.obs["Respond"] = ad_mac.obs["patient"].astype(str).map(mapping).fillna("Unknown")


rnr_col = pick_obs_col(
    ad_mac,
    ["response", "Respond", "responder", "Responder", "responder_status",
     "clinical_response", "group", "Group", "status", "RNR", "rnr"]
)
if rnr_col is None:
    raise ValueError(
        "Cannot find response column in adata.obs. "
        f"Available obs columns:\n{list(ad_mac.obs.columns)}"
    )

vals = ad_mac.obs[rnr_col].astype(str)

# strict mapping to {R, NR}
map_dict = {
    "Responder": "R",
    "Non-Responder": "NR",
    "responder": "R",
    "non-responder": "NR",
    "Non-responder": "NR",
    "NON-RESPONDER": "NR",
    "RESPONDER": "R",
}
ad_mac.obs["RNR_simplified"] = vals.map(map_dict).fillna(vals)

# assert only R/NR remain (avoid silently wrong plots)
bad = set(ad_mac.obs["RNR_simplified"].unique()) - {"R", "NR"}
if len(bad) > 0:
    raise ValueError(
        f"Found unexpected response labels after mapping: {bad}\n"
        f"Original unique labels in '{rnr_col}': {sorted(vals.unique())[:50]}"
    )

# ----------------------------
# Clustering on X_scvi
# ----------------------------
print("[3/6] Neighbors/UMAP/Leiden on scVI latent (resolution=0.6)")
rep = pick_scvi_rep(ad_mac)
if rep is None:
    raise ValueError(
        "Cannot find scVI latent representation in adata.obsm. "
        f"Available obsm keys: {list(ad_mac.obsm.keys())}"
    )
print(f"Using latent rep: {rep}")

# neighbors + umap + leiden
sc.pp.neighbors(ad_mac, use_rep=rep, n_neighbors=15, random_state=0)
sc.tl.umap(ad_mac, random_state=0)
sc.tl.leiden(ad_mac, resolution=0.6, key_added="leiden_r0.6")

# Save clustered adata
#clustered_path = os.path.join(OUTDIR, "adata_macrophage_leiden_r0.6.h5ad")
#ad_mac.write(clustered_path)
#print(f"Saved clustered adata: {clustered_path}")

# ----------------------------
# Plots: UMAP
# ----------------------------
print("[4/6] Plotting UMAPs")
# 1) by leiden
sc.pl.umap(ad_mac, color=["leiden_r0.6"], show=False)
#save_current_fig(
#    os.path.join(FIGDIR, "umap_macrophage_leiden_r0.6.png"),
#    os.path.join(FIGDIR, "umap_macrophage_leiden_r0.6.pdf"),
#)

# 2) by RNR
sc.pl.umap(ad_mac, color=["RNR_simplified"], show=False)
#save_current_fig(
#    os.path.join(FIGDIR, "umap_macrophage_RNR.png"),
#    os.path.join(FIGDIR, "umap_macrophage_RNR.pdf"),
#)

# ----------------------------
# R vs NR composition per cluster (absolute counts)
# ----------------------------
print("[5/6] Computing NR vs R counts per cluster + barplot (absolute counts)")
ct = pd.crosstab(ad_mac.obs["leiden_r0.6"], ad_mac.obs["RNR_simplified"])
ct = ct.sort_index()
ct_path = os.path.join(TABDIR, "cluster_RNR_counts_absolute.csv")
ct.to_csv(ct_path)
print(f"Saved counts table: {ct_path}")
'''
# stacked barplot (absolute counts)
ax = ct.plot(kind="bar", stacked=True, figsize=(10, 4))
ax.set_xlabel("Macrophage Leiden cluster (res=0.6)")
ax.set_ylabel("Number of cells")
ax.set_title("Macrophage clusters: NR vs R absolute cell counts")
ax.legend(title="R/NR", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
save_current_fig(
    os.path.join(FIGDIR, "barplot_cluster_RNR_absolute_stacked.png"),
    os.path.join(FIGDIR, "barplot_cluster_RNR_absolute_stacked.pdf"),
)

# side-by-side barplot (absolute counts)
ax = ct.plot(kind="bar", stacked=False, figsize=(10, 4))
ax.set_xlabel("Macrophage Leiden cluster (res=0.6)")
ax.set_ylabel("Number of cells")
ax.set_title("Macrophage clusters: NR vs R absolute cell counts")
ax.legend(title="R/NR", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
#save_current_fig(
#    os.path.join(FIGDIR, "barplot_cluster_RNR_absolute_grouped.png"),
#    os.path.join(FIGDIR, "barplot_cluster_RNR_absolute_grouped.pdf"),
#)
'''


In [ ]:
["LYZ", "CST3", "LST1", "TYROBP", "FCER1G"]

In [ ]:
sc.pl.umap(ad, color=["EPCAM", "KRT8", "PTPRB", "VWF", "COL1A1", "LYZ"], ncols=2)

In [ ]:
X_backup = ad_mac.X
ad_mac.X = np.log1p(ad_mac.layers["norm_expr"])

sc.tl.rank_genes_groups(
    ad_mac,
    groupby="leiden_r0.6",
    method="wilcoxon",
    n_genes=200,
    use_raw=False
)

# Restore X
ad_mac.X = X_backup

# Save full DE table
de_df = sc.get.rank_genes_groups_df(ad_mac, group=None)
full_de_path = os.path.join(DEDIR, "rank_genes_groups_wilcoxon_all_clusters_top200.csv")
de_df.to_csv(full_de_path, index=False)
print(f"Saved full DE: {full_de_path}")

# Save per-cluster signature genes (top50 by score)
topN = 100
clusters = sorted(ad_mac.obs["leiden_r0.6"].unique(), key=lambda x: int(x) if str(x).isdigit() else str(x))
sig_summary = []
for cl in clusters:
    df_cl = sc.get.rank_genes_groups_df(ad_mac, group=cl)
    df_cl = df_cl.dropna(subset=["names"]).copy()
    df_cl["cluster"] = cl
    out_path = os.path.join(SIGDIR, f"cluster_{cl}_signature_top{topN}.csv")
    df_cl.head(topN).to_csv(out_path, index=False)
    sig_summary.append((cl, out_path, df_cl.shape[0]))

sig_index = pd.DataFrame(sig_summary, columns=["cluster", "signature_file", "n_de_genes_returned"])
sig_index_path = os.path.join(SIGDIR, f"signature_files_index_top{topN}.csv")
sig_index.to_csv(sig_index_path, index=False)
print(f"Saved signature index: {sig_index_path}")

# Optional: quick marker dotplot to help interpret functions (edit markers as needed)
markers = {
    "Macrophage_core": ["LYZ", "CST3", "LST1", "TYROBP", "FCER1G"],
    "Inflammatory": ["IL1B", "S100A8", "S100A9", "TNF", "CXCL8"],
    "IFN": ["ISG15", "IFITM3", "IFIT1", "MX1", "OAS1"],
    "Antigen_presentation": ["HLA-DRA", "HLA-DRB1", "CD74", "HLA-DPA1", "HLA-DPB1"],
    "TAM_like": ["APOE", "C1QA", "C1QB", "C1QC", "TREM2"],
}
marker_list = [g for gs in markers.values() for g in gs]
marker_list = [g for g in marker_list if g in ad_mac.var_names]

if len(marker_list) > 0:
    sc.pl.dotplot(
        ad_mac,
        var_names=markers,
        groupby="leiden_r0.6",
        standard_scale="var",
        show=False
    )
    save_current_fig(
        os.path.join(FIGDIR, "dotplot_macrophage_markers_by_cluster.png"),
        os.path.join(FIGDIR, "dotplot_macrophage_markers_by_cluster.pdf"),
    )

print("DONE.")
print(f"Outputs written to: {OUTDIR}")


In [ ]:
ad

In [ ]:
ad.obs["majority_voting"].value_counts()

In [ ]:
import numpy as np
import scanpy as sc

# --------------------------
# 0) 生成 response 标签（cell-level）
# --------------------------
non_resp = {"PHD001", "PHD002", "PHD008"}
ad.obs["response"] = np.where(ad.obs["patient"].isin(non_resp), "NR", "R")
ad.obs["response"] = ad.obs["response"].astype("category")

# --------------------------
# 1) 定义要用的细胞集合
# --------------------------
ad_t = ad[ad.obs["majority_voting"] == "T cells"].copy()
ad_epi = ad[ad.obs["majority_voting"] == "Epithelial cells"].copy()

# myeloid = Macrophages + Monocytes (+ DC 可选)
myeloid_types = {"Macrophages", "Monocytes", "DC"}
ad_my0 = ad[ad.obs["majority_voting"].isin(myeloid_types)].copy()

# --------------------------
# 2) 先做“高置信 myeloid”保守过滤，降低 doublet/污染风险
#    （不需要病人聚合）
# --------------------------
myeloid_core = ["LYZ","C1QA","C1QB","C1QC","LST1","TYROBP","FCER1G","CTSS","CSF1R","MS4A7","CD68","APOE"]
epi_markers  = ["EPCAM","KRT8","KRT18","KRT19","MSLN","CEACAM5","TACSTD2"]
endo_markers = ["PECAM1","VWF","KDR","EMCN","PTPRB"]

def present(genes, ad_):
    return [g for g in genes if g in ad_.var_names]

myeloid_core = present(myeloid_core, ad_my0)
epi_markers  = present(epi_markers,  ad_my0)
endo_markers = present(endo_markers, ad_my0)

# score_genes 默认用 ad.X；我们用 layer=log1p_norm 更稳
# scanpy 的 score_genes 目前不直接支持 layer 参数，所以临时切换 X（最简单）
X_backup = ad_my0.X
ad_my0.X = ad_my0.layers["log1p_norm"]

sc.tl.score_genes(ad_my0, myeloid_core, score_name="score_myeloid_core")
if len(epi_markers) > 0:
    sc.tl.score_genes(ad_my0, epi_markers, score_name="score_epi")
else:
    ad_my0.obs["score_epi"] = 0.0
if len(endo_markers) > 0:
    sc.tl.score_genes(ad_my0, endo_markers, score_name="score_endo")
else:
    ad_my0.obs["score_endo"] = 0.0

ad_my0.X = X_backup

# 保守阈值：myeloid_core 高（上 50%），epi/endo 低（下 85%）
m_thr = np.quantile(ad_my0.obs["score_myeloid_core"], 0.50)
e_thr = np.quantile(ad_my0.obs["score_epi"], 0.85)
v_thr = np.quantile(ad_my0.obs["score_endo"], 0.85)

keep = (
    (ad_my0.obs["score_myeloid_core"] >= m_thr) &
    (ad_my0.obs["score_epi"] <= e_thr) &
    (ad_my0.obs["score_endo"] <= v_thr)
)
ad_my = ad_my0[keep].copy()

print("Myeloid before:", ad_my0.n_obs, "after high-confidence gating:", ad_my.n_obs)

# --------------------------
# 3) 定义“微环境轴”与“CD8命运”score
# --------------------------
# 3A) Epithelial 免疫可见性/炎症
isg = ["ISG15","IFIT1","IFIT2","IFIT3","MX1","OAS1","OAS2","IFI6","IFI27","STAT1","IRF7"]
mhc1 = ["HLA-A","HLA-B","HLA-C","B2M","TAP1","TAP2","PSMB8","PSMB9"]

# 3B) Myeloid 抑制轴 / 趋化轴
inhib = ["CD274","PDCD1LG2","IL10","TGFB1"]  # 有哪些算哪些
chem  = ["CXCL9","CXCL10","CXCL11"]

# 3C) CD8 受体/状态（你也可以换成你自己的 C3/C4 signature）
exhaust = ["PDCD1","LAG3","HAVCR2","TIGIT","CTLA4","ENTPD1","TOX"]
effector = ["NKG7","CTSW","GZMB","GZMH","PRF1","GNLY","CST7"]

def add_scores(ad_, layer, score_defs):
    Xb = ad_.X
    ad_.X = ad_.layers[layer]
    for name, genes in score_defs.items():
        genes_ = [g for g in genes if g in ad_.var_names]
        if len(genes_) >= 3:
            sc.tl.score_genes(ad_, genes_, score_name=name)
        elif len(genes_) > 0:
            # 太少就用均值（避免 score_genes 不稳定）
            # 这里用简单均值作为 fallback
            X = ad_[:, genes_].X
            ad_.obs[name] = np.asarray(X.mean(axis=1)).ravel()
        else:
            ad_.obs[name] = 0.0
    ad_.X = Xb

add_scores(ad_epi, "log1p_norm", {
    "score_epi_ISG": isg,
    "score_epi_MHCI": mhc1,
})

add_scores(ad_my, "log1p_norm", {
    "score_my_inhib": inhib,
    "score_my_chem": chem,
})

# CD8：从 T cells 里取 CD8（如果你没有 CD4/CD8 标记列，就先用全 T；最好还是再筛一下）
# 这里给一个简单 CD8 筛法：CD8A/CD8B 高于 CD4（可选）
def vec1d(x):
    """Return a dense 1D numpy array from sparse/dense/view."""
    if hasattr(x, "toarray"):
        return np.asarray(x.toarray()).ravel()
    return np.asarray(x).ravel()

if all(g in ad_t.var_names for g in ["CD8A", "CD8B", "CD4"]):
    Xb = ad_t.X
    ad_t.X = ad_t.layers["log1p_norm"]

    cd8a = vec1d(ad_t[:, "CD8A"].X)
    cd8b = vec1d(ad_t[:, "CD8B"].X)
    cd4  = vec1d(ad_t[:, "CD4"].X)

    cd8 = (cd8a + cd8b) > cd4

    ad_t.X = Xb
    ad_cd8 = ad_t[cd8].copy()
else:
    ad_cd8 = ad_t.copy()

add_scores(ad_cd8, "log1p_norm", {
    "score_cd8_exhaust": exhaust,
    "score_cd8_effector": effector,
})

# --------------------------
# 4) 画图（全是 cell-level，不画 patient 点）
# --------------------------
# Panel 1: Epithelial ISG / MHCI，按 R vs NR
sc.pl.umap(ad_epi, color=["score_epi_ISG", "score_epi_MHCI"], groupby="response",
           vmax="p99", ncols=2)

sc.pl.violin(ad_epi, keys=["score_epi_ISG","score_epi_MHCI"], groupby="response",
             stripplot=False, rotation=0)

# Panel 2: High-confidence myeloid inhibitory / chemokine niche
sc.pl.umap(ad_my, color=["score_my_inhib","score_my_chem"], groupby="response",
           vmax="p99", ncols=2)

sc.pl.violin(ad_my, keys=["score_my_inhib","score_my_chem"], groupby="response",
             stripplot=False, rotation=0)

# Panel 3: CD8 fate programs (effector vs exhaustion)
sc.pl.umap(ad_cd8, color=["score_cd8_effector","score_cd8_exhaust"], groupby="response",
           vmax="p99", ncols=2)

sc.pl.violin(ad_cd8, keys=["score_cd8_effector","score_cd8_exhaust"], groupby="response",
             stripplot=False, rotation=0)


In [ ]:
# Epithelial: ISG / MHCI，R vs NR 分开
for grp in ["R", "NR"]:
    sc.pl.umap(
        ad_epi[ad_epi.obs["response"] == grp],
        color=["score_epi_ISG", "score_epi_MHCI"],
        vmax="p99",
        ncols=2,
        title=[f"Epi ISG ({grp})", f"Epi MHC-I ({grp})"],
    )

# Myeloid: inhibitory / chem
for grp in ["R", "NR"]:
    sc.pl.umap(
        ad_my[ad_my.obs["response"] == grp],
        color=["score_my_inhib", "score_my_chem"],
        vmax="p99",
        ncols=2,
        title=[f"Myeloid inhib ({grp})", f"Myeloid chem ({grp})"],
    )

# CD8: effector / exhaustion
for grp in ["R", "NR"]:
    sc.pl.umap(
        ad_cd8[ad_cd8.obs["response"] == grp],
        color=["score_cd8_effector", "score_cd8_exhaust"],
        vmax="p99",
        ncols=2,
        title=[f"CD8 effector ({grp})", f"CD8 exhaust ({grp})"],
    )


In [ ]:
warnings.filterwarnings("ignore")

# ----------------------------
# Matplotlib config (PDF friendly)
# ----------------------------
mpl.rcParams["font.family"] = "Liberation Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42


In [ ]:
import scanpy as sc

sc.pl.violin(
    ad_epi,
    keys=["score_epi_ISG", "score_epi_MHCI"],
    groupby="response",
    stripplot=False,
    rotation=0,
)

sc.pl.violin(
    ad_my,
    keys=["score_my_inhib", "score_my_chem"],
    groupby="response",
    stripplot=False,
    rotation=0,
)

sc.pl.violin(
    ad_cd8,
    keys=["score_cd8_effector", "score_cd8_exhaust"],
    groupby="response",
    stripplot=False,
    rotation=0,
)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def to_long_df(ad_sub, feature_list, celltype_name):
    df = ad_sub.obs[["response"] + feature_list].copy()
    df["celltype"] = celltype_name
    return df.melt(id_vars=["response", "celltype"], var_name="feature", value_name="score")

df = pd.concat([
    to_long_df(ad_epi, ["score_epi_ISG", "score_epi_MHCI"], "Epithelial"),
    to_long_df(ad_my,  ["score_my_inhib", "score_my_chem"], "Myeloid (high-conf)"),
    to_long_df(ad_cd8, ["score_cd8_effector", "score_cd8_exhaust"], "CD8 T"),
], ignore_index=True)

# 顺序固定
df["response"] = pd.Categorical(df["response"], categories=["R","NR"], ordered=True)
feature_order = [
    "score_epi_ISG", "score_epi_MHCI",
    "score_my_inhib", "score_my_chem",
    "score_cd8_effector", "score_cd8_exhaust",
]
df["feature"] = pd.Categorical(df["feature"], categories=feature_order, ordered=True)

# 画图：推荐 boxplot + 不画散点（更干净）
g = sns.catplot(
    data=df,
    x="response", y="score",
    col="feature", row="celltype",
    kind="box",
    sharey=False,
    height=2.2, aspect=0.9
)

# 美化标题（可选）
for ax in g.axes.flat:
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()
